In [2]:
from sedona.spark import *
from pyspark.sql.functions import col, when, expr
#from sedona.sql.st_functions import ST_IsValid, ST_IsValidReason, ST_MakeValid DEPRECATED
from sedona.spark.sql.st_functions import ST_IsValid, ST_IsValidReason, ST_MakeValid
from pyspark.sql import DataFrame
import urllib.request
import json

In [3]:
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/02 19:23:01 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

## Grab Area of Interest

In [4]:
import wkls

# seattle_dt = 'POLYGON ((-122.354307 47.612065, -122.318087 47.612065, -122.318087 47.626645, -122.354307 47.626645, -122.354307 47.612065))'

washington = wkls.us.wa.wkt()
seattle = wkls.us.wa.seattle.wkt()
kirkland = wkls.us.wa.kirkland.wkt()
bellevue = wkls.us.wa.bellevue.wkt()

# Small bounding box in central Kirkland (~0.5km²)
kirkland_small = "POLYGON((-122.212 47.676, -122.200 47.676, -122.200 47.682, -122.212 47.682, -122.212 47.676))"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Grab residential Buildings

The Spatial Filter gets pushed down to storage level to only grab residential buildings that intersect the AOI

In [5]:
buildings_df = sedona.table("wherobots_open_data.overture_maps_foundation.buildings_building")\
                .filter(f"""
                    ST_Intersects(geometry, ST_GeomFromWKT('{kirkland_small}'))
                    AND subtype == 'residential'
                """)
buildings_df.createOrReplaceTempView("buildings")
buildings_df.show(3)
buildings_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+-----+-----------+--------+-----------------+-----+---------+--------------+----------+----------------------+----------+---------+------------+---------------+-------------+----------+--------------+----------------+----------+-----------+
|                  id|            geometry|                bbox|version|             sources|level|    subtype|   class|           height|names|has_parts|is_underground|num_floors|num_floors_underground|min_height|min_floor|facade_color|facade_material|roof_material|roof_shape|roof_direction|roof_orientation|roof_color|roof_height|
+--------------------+--------------------+--------------------+-------+--------------------+-----+-----------+--------+-----------------+-----+---------+--------------+----------+----------------------+----------+---------+------------+---------------+-------------+----------+--------------+----------------+----------+-----------

77

## Grab Overture Places with pregenerated Drive time Isochrones

Same Spatial filter pushdown concept with Places of Interest (POI) dataset.

Here, we apply a buffer of 20KM to the AOI to make sure buildings near the edge of the AOI have coverage of POIs on all sides

In [6]:
poi_isochones_df = sedona.table("wherobots_pro_data.overture_maps_foundation.overture_places_with_isochrones")\
                .filter(f"""
                    ST_Intersects(
                        geometry, 
                        ST_Buffer(ST_GeomFromWKT('{kirkland_small}'), 20000, true)
                    )
                """)
poi_isochones_df.createOrReplaceTempView("poi_iso")
poi_isochones_df.show(3)
poi_isochones_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+------+--------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|                  id|            geometry|                bbox|version|             sources|               names|          categories|        confidence|            websites|             socials|emails|        phones|brand|           addresses|      isochrone_5min|     isochrone_10min|     isochrone_15min|     isochrone_20min|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+------+--------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|08f28d552

69910

In [7]:
history_museums_df = poi_isochones_df.filter("categories.primary ILIKE '%history_museum%'")
history_museums_df.createOrReplaceTempView("museum")
history_museums_df.show(3)
history_museums_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+------+--------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|                  id|            geometry|                bbox|version|             sources|               names|          categories|         confidence|            websites|             socials|emails|        phones|brand|           addresses|      isochrone_5min|     isochrone_10min|     isochrone_15min|     isochrone_20min|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+------+--------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|08f28d

20

## Enriching Residential Buildings with 5-Minute Museum Accessibility via car

In [8]:
%time

res = sedona.sql("""
    SELECT
      b.id,
      FIRST(b.geometry) AS geometry,
      FIRST(b.subtype) AS subtype,
      FIRST(b.class) AS class,
      FIRST(b.names.primary) AS name,
      (SUM(CASE WHEN m.id IS NOT NULL THEN 1 ELSE 0 END) > 0) AS has_museum_5min
    FROM buildings b
    LEFT JOIN museum m
      ON ST_Intersects(b.geometry, m.isochrone_5min)
    GROUP BY b.id
""")

res.cache().count()

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 4.29 μs


77

In [9]:
res.show(5)

+--------------------+--------------------+-----------+--------+----+---------------+
|                  id|            geometry|    subtype|   class|name|has_museum_5min|
+--------------------+--------------------+-----------+--------+----+---------------+
|c7aa4d1a-b5c7-413...|POLYGON ((-122.20...|residential|detached|NULL|          false|
|26542586-8c1a-489...|POLYGON ((-122.20...|residential|detached|NULL|          false|
|8e88db80-d0b1-464...|POLYGON ((-122.20...|residential|   house|NULL|          false|
|fedcaa4d-2417-46e...|POLYGON ((-122.20...|residential|detached|NULL|          false|
|c21f2eba-174c-487...|POLYGON ((-122.21...|residential|detached|NULL|          false|
+--------------------+--------------------+-----------+--------+----+---------------+
only showing top 5 rows


In [10]:
# Create a new Havasu (Iceberg) database

database = 'gde_silver'

sedona.sql(f'CREATE DATABASE IF NOT EXISTS org_catalog.{database}')
database

'gde_silver'

In [11]:
%%time

res.writeTo(f"org_catalog.{database}.residential_buildings_near_museum_silver").createOrReplace()

CPU times: user 7.13 ms, sys: 1.32 ms, total: 8.45 ms
Wall time: 3.72 s


In [12]:
sedona.sql("SHOW TABLES IN org_catalog.gde_silver").show()

+----------+--------------------+-----------+
| namespace|           tableName|isTemporary|
+----------+--------------------+-----------+
|gde_silver|         census_data|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver| homes_flood_hazards|      false|
|gde_silver| homes_school_scores|      false|
|gde_silver|homes_seismic_haz...|      false|
|gde_silver|house_sales_ndvi_...|      false|
|gde_silver|house_sales_tri_s...|      false|
|gde_silver|house_sales_zonal...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|     roads_proximity|      false|
|gde_silver|         school_data|      false|
+----------+--------------------+-----------+

